# Planners-6-Domains - Domaines Classiques de Planification

**Navigation** : [Index](../../README.md) | [<< Heuristiques](Planners-5-Heuristics.ipynb) | [OR-Tools >>](../03-Advanced/Planners-7-OR-Tools.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Modeliser** le domaine Block World en PDDL
2. **Modeliser** le domaine Logistics avec transport multi-agent
3. **Resoudre** ces problemes avec Fast Downward
4. **Analyser** la qualite des plans et comparer les domaines
5. **Connaitre** d'autres domaines classiques (Gripper, Ferry, Hanoi)

### Prerequis
- Avoir suivi [Planners-2-PDDL-Basics](../01-Foundation/Planners-2-PDDL-Basics.ipynb)
- Connaissance de la syntaxe PDDL
- unified-planning installe

### Duree estimee : 50 minutes

---

## 1. Introduction aux domaines classiques

Les **domaines classiques** de planification sont des benchmarks standardises utilises depuis les premieres competitions IPC (International Planning Competition) en 1998. Ils permettent de comparer les performances des planificateurs sur des problemes bien definis.

### 1.1 Taxonomie des domaines

| Domaine | Type | Complexite | Usage principal |
|---------|------|------------|----------------|
| **Block World** | Manipulation | Simple | Tutoriel, demo |
| **Gripper** | Transport simple | Simple | Test algorithmes |
| **Logistics** | Transport multi-agent | Moyen | Applications reelles |
| **Depots** | Gestion stock | Moyen | Planification industrielle |
| **Ferry** | Transport sequentiel | Simple | Pedagogie |
| **Hanoi** | Puzzle | Moyen | Test recursive |

### 1.2 Pourquoi etudier ces domaines ?

1. **Standardisation** : Benchmarks reconnus par la communaute
2. **Progression pedagogique** : Du simple au complexe
3. **Applications reelles** : Logistics = livraison, Depots = entrepot
4. **Comparaison** : Performance relative des heuristiques

In [1]:
# Verification de l'environnement
import os
import sys

try:
    import unified_planning as up
    from unified_planning.shortcuts import *
    print(f"unified-planning version: {up.__version__}")
    UP_AVAILABLE = True
except ImportError:
    print("ERREUR: unified-planning non installe")
    print("pip install unified-planning")
    UP_AVAILABLE = False

# Configuration pour l'affichage
import warnings
warnings.filterwarnings('ignore')

unified-planning version: 1.3.0


---

## 2. Block World - Le domaine de reference

Le **Block World** (monde des blocs) est le domaine le plus etudie en planification. Un bras robotique manipule des blocs pour construire des tours.

### 2.1 Description du domaine

**Elements du monde** :
- **Blocs** : Objets manipulables
- **Table** : Surface de base (infinie)
- **Bras robotique** : Peut tenir un bloc a la fois

**Actions disponibles** :
| Action | Preconditions | Effets |
|--------|---------------|--------|
| `pick(x)` | `clear(x)`, `ontable(x)`, `handempty` | `holding(x)` |
| `putdown(x)` | `holding(x)` | `ontable(x)`, `clear(x)`, `handempty` |
| `stack(x,y)` | `holding(x)`, `clear(y)` | `on(x,y)`, `clear(x)`, `handempty` |
| `unstack(x,y)` | `on(x,y)`, `clear(x)`, `handempty` | `holding(x)`, `clear(y)` |

### 2.2 Domaine PDDL Block World

In [2]:
# Domaine PDDL - Block World
BLOCKS_DOMAIN = """
(define (domain blocks)
  (:requirements :strips :typing)
  (:types block)
  
  (:predicates
    ;; Relations spatiales
    (on ?x ?y - block)       ;; x est sur y
    (ontable ?x - block)     ;; x est sur la table
    (clear ?x - block)       ;; rien n'est sur x
    
    ;; Etat du bras
    (holding ?x - block)     ;; le bras tient x
    (handempty)              ;; le bras est vide
  )
  
  ;; Action : Prendre un bloc de la table
  (:action pick-up
    :parameters (?x - block)
    :precondition (and (clear ?x) (ontable ?x) (handempty))
    :effect (and 
              (holding ?x) 
              (not (ontable ?x)) 
              (not (clear ?x)) 
              (not (handempty)))
  )
  
  ;; Action : Poser un bloc sur la table
  (:action put-down
    :parameters (?x - block)
    :precondition (holding ?x)
    :effect (and 
              (ontable ?x) 
              (clear ?x) 
              (handempty) 
              (not (holding ?x)))
  )
  
  ;; Action : Empiler un bloc sur un autre
  (:action stack
    :parameters (?x ?y - block)
    :precondition (and (holding ?x) (clear ?y))
    :effect (and 
              (on ?x ?y) 
              (clear ?x) 
              (handempty) 
              (not (holding ?x)) 
              (not (clear ?y)))
  )
  
  ;; Action : Deseempiler un bloc d'un autre
  (:action unstack
    :parameters (?x ?y - block)
    :precondition (and (on ?x ?y) (clear ?x) (handempty))
    :effect (and 
              (holding ?x) 
              (clear ?y) 
              (not (on ?x ?y)) 
              (not (clear ?x)) 
              (not (handempty)))
  )
)
"""

print("Domaine PDDL Block World defini")
print("Actions: pick-up, put-down, stack, unstack")

Domaine PDDL Block World defini
Actions: pick-up, put-down, stack, unstack


### 2.3 Problemes Block World

In [3]:
# Probleme simple : construire une tour de 3 blocs
BLOCKS_PROBLEM_TOWER = """
(define (problem blocks-tower)
  (:domain blocks)
  (:objects a b c - block)
  
  (:init
    ;; Tous les blocs sont sur la table
    (ontable a) (ontable b) (ontable c)
    ;; Tous sont libres
    (clear a) (clear b) (clear c)
    ;; Le bras est vide
    (handempty)
  )
  
  (:goal (and 
    (on a b)   ;; a sur b
    (on b c)   ;; b sur c
  ))
)
"""

print("Probleme 'blocks-tower' : Construire une tour c <- b <- a")
print("\nEtat initial: a, b, c sur la table")
print("But: a sur b, b sur c (tour de 3)")

Probleme 'blocks-tower' : Construire une tour c <- b <- a

Etat initial: a, b, c sur la table
But: a sur b, b sur c (tour de 3)


Definition d'un probleme Block World plus complexe impliquant l'inversion de deux tours, ce qui necessite de defaire une configuration existante avant de reconstruire.

In [4]:
# Probleme plus complexe : inverser deux tours
BLOCKS_PROBLEM_REVERSE = """
(define (problem blocks-reverse)
  (:domain blocks)
  (:objects a b c d - block)
  
  (:init
    ;; Tour 1 : d sur c (c sur table)
    (ontable c) (on d c) (clear d)
    ;; Tour 2 : b sur a (a sur table)
    (ontable a) (on b a) (clear b)
    ;; Bras vide
    (handempty)
  )
  
  (:goal (and 
    ;; Inverser : c sur d, a sur b
    (ontable d) (on c d)
    (ontable b) (on a b)
  ))
)
"""

print("Probleme 'blocks-reverse' : Inverser deux tours")
print("\nEtat initial:")
print("  Tour 1: table <- c <- d")
print("  Tour 2: table <- a <- b")
print("\nBut:")
print("  Tour 1: table <- d <- c")
print("  Tour 2: table <- b <- a")

Probleme 'blocks-reverse' : Inverser deux tours

Etat initial:
  Tour 1: table <- c <- d
  Tour 2: table <- a <- b

But:
  Tour 1: table <- d <- c
  Tour 2: table <- b <- a


### 2.4 Resolution avec unified-planning

In [5]:
if UP_AVAILABLE:
    # Definition du domaine Block World avec unified-planning
    from collections import OrderedDict
    
    Block = UserType('Block')
    
    # Predicats - IMPORTANT: utiliser OrderedDict pour le signature (API v1.3+)
    on = Fluent('on', BoolType(), OrderedDict([('x', Block), ('y', Block)]))
    ontable = Fluent('ontable', BoolType(), OrderedDict([('x', Block)]))
    clear = Fluent('clear', BoolType(), OrderedDict([('x', Block)]))
    holding = Fluent('holding', BoolType(), OrderedDict([('x', Block)]))
    handempty = Fluent('handempty', BoolType())
    
    # Variables pour les actions
    x = Variable('x', Block)
    y = Variable('y', Block)
    
    # Action pick-up
    pick_up = InstantaneousAction('pick-up', OrderedDict([('x', Block)]))
    x_pu = pick_up.parameter('x')  # IMPORTANT: utiliser action.parameter()
    pick_up.add_precondition(clear(x_pu))
    pick_up.add_precondition(ontable(x_pu))
    pick_up.add_precondition(handempty)
    pick_up.add_effect(holding(x_pu), True)
    pick_up.add_effect(ontable(x_pu), False)
    pick_up.add_effect(clear(x_pu), False)
    pick_up.add_effect(handempty, False)
    
    # Action put-down
    put_down = InstantaneousAction('put-down', OrderedDict([('x', Block)]))
    x_pd = put_down.parameter('x')
    put_down.add_precondition(holding(x_pd))
    put_down.add_effect(ontable(x_pd), True)
    put_down.add_effect(clear(x_pd), True)
    put_down.add_effect(handempty, True)
    put_down.add_effect(holding(x_pd), False)
    
    # Action stack
    stack = InstantaneousAction('stack', OrderedDict([('x', Block), ('y', Block)]))
    x_s = stack.parameter('x')
    y_s = stack.parameter('y')
    stack.add_precondition(holding(x_s))
    stack.add_precondition(clear(y_s))
    stack.add_effect(on(x_s, y_s), True)
    stack.add_effect(clear(x_s), True)
    stack.add_effect(handempty, True)
    stack.add_effect(holding(x_s), False)
    stack.add_effect(clear(y_s), False)
    
    # Action unstack
    unstack = InstantaneousAction('unstack', OrderedDict([('x', Block), ('y', Block)]))
    x_u = unstack.parameter('x')
    y_u = unstack.parameter('y')
    unstack.add_precondition(on(x_u, y_u))
    unstack.add_precondition(clear(x_u))
    unstack.add_precondition(handempty)
    unstack.add_effect(holding(x_u), True)
    unstack.add_effect(clear(y_u), True)
    unstack.add_effect(on(x_u, y_u), False)
    unstack.add_effect(clear(x_u), False)
    unstack.add_effect(handempty, False)
    
    # Creation du probleme
    blocks_problem = Problem('blocks-tower')
    
    # Objets
    a = Object('a', Block)
    b = Object('b', Block)
    c = Object('c', Block)
    blocks_problem.add_objects([a, b, c])
    
    # Etat initial
    blocks_problem.set_initial_value(ontable(a), True)
    blocks_problem.set_initial_value(ontable(b), True)
    blocks_problem.set_initial_value(ontable(c), True)
    blocks_problem.set_initial_value(clear(a), True)
    blocks_problem.set_initial_value(clear(b), True)
    blocks_problem.set_initial_value(clear(c), True)
    blocks_problem.set_initial_value(handempty, True)
    
    # But : a sur b, b sur c
    blocks_problem.add_goal(on(a, b))
    blocks_problem.add_goal(on(b, c))
    
    # Ajout des actions
    blocks_problem.add_action(pick_up)
    blocks_problem.add_action(put_down)
    blocks_problem.add_action(stack)
    blocks_problem.add_action(unstack)
    
    print("Probleme Block World cree avec unified-planning")
    print(f"Objets: {[o.name for o in blocks_problem.all_objects]}")
    print(f"Actions: {[a.name for a in blocks_problem.actions]}")

Probleme Block World cree avec unified-planning
Objets: ['a', 'b', 'c']
Actions: ['pick-up', 'put-down', 'stack', 'unstack']


Resolution du probleme Block World avec pyperplan et affichage du plan trouve, en verifiant que le planificateur produce une solution valide pour les deux instances.

In [6]:
if UP_AVAILABLE:
    from unified_planning.engines import PlanGenerationResultStatus
    
    try:
        with OneshotPlanner(name='pyperplan') as planner:
            print(f"Planificateur: {planner.name}")
            print("Resolution du probleme Block World...\n")
            
            result = planner.solve(blocks_problem)
            
            if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
                print("Solution trouvee !")
                print("=" * 50)
                for i, action in enumerate(result.plan.actions):
                    params = ', '.join(str(p.object()) for p in action.actual_parameters)
                    print(f"  {i+1}. {action.action.name}({params})")
                print("=" * 50)
                print(f"Longueur du plan: {len(result.plan.actions)} actions")
            else:
                print(f"Statut: {result.status}")
    except Exception as e:
        print(f"Erreur: {e}")
        print("\nSolution theorique:")
        print("  1. pick-up(b)")
        print("  2. put-down(b) [sur c] -> stack(b, c)")
        print("  3. pick-up(a)")
        print("  4. stack(a, b)")

NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 563 of `C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\unified_planning\shortcuts.py`, you are using the following planning engine:
  * Engine name: pyperplan
  * Developers:  Albert-Ludwigs-Universität Freiburg (Yusra Alkhazraji, Matthias Frorath, Markus Grützner, Malte Helmert, Thomas Liebetraut, Robert Mattmüller, Manuela Ortlieb, Jendrik Seipp, Tobias Springenberg, Philip Stahl, Jan Wülfing)
  * Description: Pyperplan is a lightweight STRIPS planner written in Python.

Planificateur: Pyperplan
Resolution du probleme Block World...

Solution trouvee !
  1. pick-up(b)
  2. stack(b, c)
  3. pick-up(a)
  4. stack(a, b)
Longueur du plan: 4 actions


### Interpretation du plan Block World

**Analyse de la solution** :

| Etape | Action | Effet sur l'etat |
|-------|--------|------------------|
| 0 | (initial) | a,b,c sur table |
| 1 | pick-up(b) | holding(b) |
| 2 | stack(b,c) | b sur c |
| 3 | pick-up(a) | holding(a) |
| 4 | stack(a,b) | a sur b (but atteint) |

> **Note** : Le plan optimal a 4 actions. La construction se fait de bas en haut (d'abord b sur c, puis a sur b).

---

## 3. Domaine Logistics - Transport multi-agent

Le domaine **Logistics** modelise le transport de colis entre lieux avec differents types de vehicules. C'est un benchmark classique plus realiste que Block World.

### 3.1 Description du domaine

**Elements du monde** :
- **Colis (packages)** : Objets a transporter
- **Vehicules** : Camions (routes) et avions (aeroports)
- **Lieux** : Positions avec connexions

**Hierarchie des types** :
```
object
  |-- location (place, city)
  |-- vehicle (truck, airplane)
  |-- package
```

### 3.2 Domaine PDDL Logistics complet

In [7]:
# Domaine PDDL - Logistics complet
LOGISTICS_DOMAIN = """
(define (domain logistics)
  (:requirements :strips :typing)
  (:types 
    location - object
    city place - location
    vehicle - object
    truck airplane - vehicle
    package - object
  )
  
  (:predicates
    ;; Position des objets
    (at ?p - package ?l - location)
    (in ?p - package ?v - vehicle)
    (at-vehicle ?v - vehicle ?l - location)
    
    ;; Connexions
    (in-city ?l - place ?c - city)
    
    ;; Aeroports
    (airport ?l - location)
  )
  
  ;; Action : Charger un colis
  (:action load
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (and (at ?p ?l) (at-vehicle ?v ?l))
    :effect (and 
              (in ?p ?v) 
              (not (at ?p ?l)))
  )
  
  ;; Action : Decharger un colis
  (:action unload
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (and (in ?p ?v) (at-vehicle ?v ?l))
    :effect (and 
              (at ?p ?l) 
              (not (in ?p ?v)))
  )
  
  ;; Action : Conduire un camion
  (:action drive
    :parameters (?t - truck ?from ?to - place)
    :precondition (and 
                    (at-vehicle ?t ?from) 
                    (in-city ?from ?c) 
                    (in-city ?to ?c))
    :effect (and 
              (at-vehicle ?t ?to) 
              (not (at-vehicle ?t ?from)))
  )
  
  ;; Action : Faire voler un avion
  (:action fly
    :parameters (?a - airplane ?from ?to - location)
    :precondition (and 
                    (at-vehicle ?a ?from) 
                    (airport ?from) 
                    (airport ?to))
    :effect (and 
              (at-vehicle ?a ?to) 
              (not (at-vehicle ?a ?from)))
  )
)
"""

print("Domaine PDDL Logistics defini")
print("Actions: load, unload, drive, fly")

Domaine PDDL Logistics defini
Actions: load, unload, drive, fly


### 3.3 Probleme Logistics multi-ville

In [8]:
# Probleme Logistics avec 2 villes
LOGISTICS_PROBLEM = """
(define (problem logistics-2cities)
  (:domain logistics)
  
  (:objects
    ;; Villes
    paris lyon - city
    
    ;; Lieux
    paris-airport paris-center - place
    lyon-airport lyon-center - place
    
    ;; Vehicules
    truck1 - truck
    plane1 - airplane
    
    ;; Colis
    pkg1 pkg2 - package
  )
  
  (:init
    ;; Appartenance aux villes
    (in-city paris-airport paris)
    (in-city paris-center paris)
    (in-city lyon-airport lyon)
    (in-city lyon-center lyon)
    
    ;; Aeroports
    (airport paris-airport)
    (airport lyon-airport)
    
    ;; Positions initiales
    (at pkg1 paris-center)
    (at pkg2 lyon-center)
    (at-vehicle truck1 paris-center)
    (at-vehicle plane1 paris-airport)
  )
  
  (:goal (and
    ;; pkg1 doit aller a lyon-center
    (at pkg1 lyon-center)
    ;; pkg2 doit aller a paris-center
    (at pkg2 paris-center)
  ))
)
"""

print("Probleme 'logistics-2cities' : Echange de colis entre Paris et Lyon")
print("\nScenario:")
print("  pkg1: Paris-center -> Lyon-center")
print("  pkg2: Lyon-center -> Paris-center")
print("\nVehicules disponibles:")
print("  truck1: camion a Paris-center")
print("  plane1: avion a Paris-airport")

Probleme 'logistics-2cities' : Echange de colis entre Paris et Lyon

Scenario:
  pkg1: Paris-center -> Lyon-center
  pkg2: Lyon-center -> Paris-center

Vehicules disponibles:
  truck1: camion a Paris-center
  plane1: avion a Paris-airport


### 3.4 Resolution avec unified-planning

In [9]:
if UP_AVAILABLE:
    from collections import OrderedDict
    
    # Definition du domaine Logistics
    Location = UserType('Location')
    City = UserType('City', father=Location)
    Place = UserType('Place', father=Location)
    Vehicle = UserType('Vehicle')
    Truck = UserType('Truck', father=Vehicle)
    Airplane = UserType('Airplane', father=Vehicle)
    Package = UserType('Package')
    
    # Predicats - IMPORTANT: utiliser OrderedDict pour le signature (API v1.3+)
    at_pkg = Fluent('at', BoolType(), OrderedDict([('p', Package), ('l', Location)]))
    in_pkg = Fluent('in', BoolType(), OrderedDict([('p', Package), ('v', Vehicle)]))
    at_veh = Fluent('at-vehicle', BoolType(), OrderedDict([('v', Vehicle), ('l', Location)]))
    # Note: in-city requires specific city objects, not used in simplified model
    is_airport = Fluent('airport', BoolType(), OrderedDict([('l', Location)]))
    
    # Variables
    p = Variable('p', Package)
    v = Variable('v', Vehicle)
    l = Variable('l', Location)
    
    # Actions
    load = InstantaneousAction('load', OrderedDict([('p', Package), ('v', Vehicle), ('l', Location)]))
    p_l = load.parameter('p')
    v_l = load.parameter('v')
    l_l = load.parameter('l')
    load.add_precondition(at_pkg(p_l, l_l))
    load.add_precondition(at_veh(v_l, l_l))
    load.add_effect(in_pkg(p_l, v_l), True)
    load.add_effect(at_pkg(p_l, l_l), False)
    
    unload = InstantaneousAction('unload', OrderedDict([('p', Package), ('v', Vehicle), ('l', Location)]))
    p_u = unload.parameter('p')
    v_u = unload.parameter('v')
    l_u = unload.parameter('l')
    unload.add_precondition(in_pkg(p_u, v_u))
    unload.add_precondition(at_veh(v_u, l_u))
    unload.add_effect(at_pkg(p_u, l_u), True)
    unload.add_effect(in_pkg(p_u, v_u), False)
    
    # Drive action - simplified without city constraint
    drive = InstantaneousAction('drive', OrderedDict([('v', Vehicle), ('from', Location), ('to', Location)]))
    v_d = drive.parameter('v')
    from_d = drive.parameter('from')
    to_d = drive.parameter('to')
    drive.add_precondition(at_veh(v_d, from_d))
    drive.add_effect(at_veh(v_d, to_d), True)
    drive.add_effect(at_veh(v_d, from_d), False)
    
    # Fly action
    fly = InstantaneousAction('fly', OrderedDict([('a', Airplane), ('from', Location), ('to', Location)]))
    a_f = fly.parameter('a')
    from_f = fly.parameter('from')
    to_f = fly.parameter('to')
    fly.add_precondition(at_veh(a_f, from_f))
    fly.add_precondition(is_airport(from_f))
    fly.add_precondition(is_airport(to_f))
    fly.add_effect(at_veh(a_f, to_f), True)
    fly.add_effect(at_veh(a_f, from_f), False)
    
    print("Actions Logistics definies: load, unload, drive, fly")

Actions Logistics definies: load, unload, drive, fly


Creation du probleme Logistics concret avec les objets (colis, vehicules, locations), l'etat initial et le but, en utilisant l'API unified-planning.

In [10]:
if UP_AVAILABLE:
    from collections import OrderedDict
    
    # Creation du probleme simplifie
    logistics_problem = Problem('logistics-simple')
    
    # Objets
    depot = Object('depot', Location)
    warehouse = Object('warehouse', Location)
    store = Object('store', Location)
    box1 = Object('box1', Package)
    box2 = Object('box2', Package)
    truck1 = Object('truck1', Truck)
    
    logistics_problem.add_objects([depot, warehouse, store, box1, box2, truck1])
    
    # Etat initial
    logistics_problem.set_initial_value(at_pkg(box1, depot), True)
    logistics_problem.set_initial_value(at_pkg(box2, depot), True)
    logistics_problem.set_initial_value(at_veh(truck1, depot), True)
    
    # But
    logistics_problem.add_goal(at_pkg(box1, store))
    logistics_problem.add_goal(at_pkg(box2, warehouse))
    
    # Ajout des actions (version simplifiee sans types hierarchiques)
    logistics_problem.add_action(load)
    logistics_problem.add_action(unload)
    logistics_problem.add_action(drive)
    
    print("Probleme Logistics simplifie cree")

Probleme Logistics simplifie cree


Lancement du planificateur sur le probleme Logistics et affichage du plan genere avec les actions de chargement, deplacement et dechargement des colis.

In [11]:
if UP_AVAILABLE:
    try:
        with OneshotPlanner(name='pyperplan') as planner:
            print(f"Planificateur: {planner.name}")
            print("Resolution du probleme Logistics...\n")
            
            result = planner.solve(logistics_problem)
            
            if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
                print("Solution trouvee !")
                print("=" * 50)
                for i, action in enumerate(result.plan.actions):
                    params = ', '.join(str(p.object()) for p in action.actual_parameters)
                    print(f"  {i+1}. {action.action.name}({params})")
                print("=" * 50)
                print(f"Longueur du plan: {len(result.plan.actions)} actions")
            else:
                print(f"Statut: {result.status}")
    except Exception as e:
        print(f"Note: {e}")
        print("\nSolution theorique pour le probleme complet:")
        print("  1. load(pkg1, truck1, paris-center)")
        print("  2. drive(truck1, paris-center, paris-airport)")
        print("  3. unload(pkg1, truck1, paris-airport)")
        print("  4. load(pkg1, plane1, paris-airport)")
        print("  5. fly(plane1, paris-airport, lyon-airport)")
        print("  6. unload(pkg1, plane1, lyon-airport)")
        print("  7. load(pkg1, truck2, lyon-airport)")
        print("  8. drive(truck2, lyon-airport, lyon-center)")
        print("  9. unload(pkg1, truck2, lyon-center)")
        print("  (Similaire pour pkg2 dans l'autre sens)")

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 563 of `C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\unified_planning\shortcuts.py`, you are using the following planning engine:
  * Engine name: pyperplan
  * Developers:  Albert-Ludwigs-Universität Freiburg (Yusra Alkhazraji, Matthias Frorath, Markus Grützner, Malte Helmert, Thomas Liebetraut, Robert Mattmüller, Manuela Ortlieb, Jendrik Seipp, Tobias Springenberg, Philip Stahl, Jan Wülfing)
  * Description: Pyperplan is a lightweight STRIPS planner written in Python.

Planificateur: Pyperplan
Resolution du probleme Logistics...

Solution trouvee !
  1. load(box2, truck1, depot)
  2. load(box1, truck1, depot)
  3. drive(truck1, depot, warehouse)
  4. unload(box2, truck1, warehouse)
  5. drive(truck1, warehouse, store)
  6. unload(box1, truck1, store)
Longueur du plan: 6 actions


### Interpretation du plan Logistics

**Analyse de la complexite** :

| Aspect | Valeur | Observation |
|--------|--------|-------------|
| **Actions requises** | ~9 par colis | load/drive/unload pour chaque transfert |
| **Coordination** | Necessaire | L'avion doit etre disponible |
| **Parallelisme possible** | Oui | pkg1 et pkg2 peuvent etre traites en parallele |

> **Note** : Le domaine Logistics illustre la necessite de coordonner plusieurs vehicules. La planification optimale minimise le temps total, pas seulement le nombre d'actions.

---

## 4. Autres domaines classiques

En plus de Block World et Logistics, plusieurs autres domaines sont etudies en planification classique.

### 4.1 Domaine Gripper

Un robot avec deux pinces doit deplacer des balles entre deux pieces.

In [12]:
# Domaine PDDL - Gripper (extrait)
GRIPPER_DOMAIN = """
(define (domain gripper)
  (:requirements :strips :typing)
  (:types room ball gripper)
  
  (:predicates
    (at-robby ?r - room)        ;; position du robot
    (at ?b - ball ?r - room)    ;; position d'une balle
    (free ?g - gripper)         ;; pince libre
    (carry ?b - ball ?g - gripper)  ;; balle tenue
  )
  
  (:action move
    :parameters (?from ?to - room)
    :precondition (at-robby ?from)
    :effect (and (at-robby ?to) (not (at-robby ?from)))
  )
  
  (:action pick
    :parameters (?b - ball ?r - room ?g - gripper)
    :precondition (and (at ?b ?r) (at-robby ?r) (free ?g))
    :effect (and (carry ?b ?g) (not (at ?b ?r)) (not (free ?g)))
  )
  
  (:action drop
    :parameters (?b - ball ?r - room ?g - gripper)
    :precondition (and (carry ?b ?g) (at-robby ?r))
    :effect (and (at ?b ?r) (free ?g) (not (carry ?b ?g)))
  )
)
"""

print("Domaine Gripper : Robot avec 2 pinces")
print("Caracteristique : Le robot peut porter 2 balles a la fois !")

Domaine Gripper : Robot avec 2 pinces
Caracteristique : Le robot peut porter 2 balles a la fois !


### 4.2 Domaine Ferry

Un ferry transporte des voitures entre deux rives.

In [13]:
# Domaine PDDL - Ferry
FERRY_DOMAIN = """
(define (domain ferry)
  (:requirements :strips :typing)
  (:types location car)
  
  (:predicates
    (at-ferry ?l - location)     ;; position du ferry
    (at-car ?c - car ?l - location)  ;; position d'une voiture
    (empty-ferry)                ;; ferry vide
    (on ?c - car)                ;; voiture sur le ferry
  )
  
  (:action board
    :parameters (?c - car ?l - location)
    :precondition (and (at-car ?c ?l) (at-ferry ?l) (empty-ferry))
    :effect (and (on ?c) (not (at-car ?c ?l)) (not (empty-ferry)))
  )
  
  (:action sail
    :parameters (?from ?to - location)
    :precondition (at-ferry ?from)
    :effect (and (at-ferry ?to) (not (at-ferry ?from)))
  )
  
  (:action debark
    :parameters (?c - car ?l - location)
    :precondition (and (on ?c) (at-ferry ?l))
    :effect (and (at-car ?c ?l) (empty-ferry) (not (on ?c)))
  )
)
"""

print("Domaine Ferry : Transport sequentiel")
print("Contrainte : Le ferry ne peut porter qu'une voiture a la fois")

Domaine Ferry : Transport sequentiel
Contrainte : Le ferry ne peut porter qu'une voiture a la fois


### 4.3 Domaine Hanoi (Tours de Hanoi)

Le probleme classique des tours de Hanoi.

In [14]:
# Domaine PDDL - Hanoi
HANOI_DOMAIN = """
(define (domain hanoi)
  (:requirements :strips :typing)
  (:types peg disk)
  
  (:predicates
    (on ?d1 ?d2 - disk)       ;; disque d1 sur d2
    (on-peg ?d - disk ?p - peg)  ;; disque sur le peg
    (clear ?d - disk)         ;; rien sur le disque
    (smaller ?d1 ?d2 - disk)  ;; d1 plus petit que d2
    (peg-clear ?p - peg)      ;; peg vide (ou plus grand disque)
  )
  
  (:action move
    :parameters (?d - disk ?from ?to - disk)
    :precondition (and 
                    (clear ?d) 
                    (clear ?to) 
                    (smaller ?d ?to) 
                    (on ?d ?from))
    :effect (and 
              (on ?d ?to) 
              (clear ?from) 
              (not (on ?d ?from)) 
              (not (clear ?to)))
  )
  
  (:action move-to-peg
    :parameters (?d - disk ?from - disk ?to - peg)
    :precondition (and (clear ?d) (on ?d ?from) (peg-clear ?to))
    :effect (and 
              (on-peg ?d ?to) 
              (clear ?from) 
              (not (on ?d ?from)))
  )
)
"""

print("Domaine Hanoi : Tours de Hanoi")
print("Complexite : 2^n - 1 mouvements pour n disques")

Domaine Hanoi : Tours de Hanoi
Complexite : 2^n - 1 mouvements pour n disques


### 4.4 Comparaison des domaines

| Domaine | Type | Complexite etats | Complexite plans | Patron principal |
|---------|------|------------------|------------------|------------------|
| **Block World** | Manipulation | O(n^2) | O(n^2) | Empiler/Deseempiler |
| **Gripper** | Transport | O(n^2) | O(n) | Collecter/Deposer |
| **Ferry** | Transport | O(n) | O(n^2) | Sequencement |
| **Logistics** | Multi-agent | O(n^m) | Variable | Coordination |
| **Hanoi** | Puzzle | O(3^n) | O(2^n) | Recursivite |

---

## 5. Analyse comparative des domaines

Cette section analyse les caracteristiques structurelles des differents domaines.

In [15]:
# Analyse comparative des domaines
domains_analysis = {
    'Block World': {
        'objects_type': 'Homogene (blocs)',
        'actions_per_object': 4,
        'state_size_formula': 'O(n^2)',
        'plan_length_typical': 'O(n^2)',
        'key_challenge': 'Ordre des empilages'
    },
    'Gripper': {
        'objects_type': 'Mixte (balles, pinces)',
        'actions_per_object': 3,
        'state_size_formula': 'O(n x k)',
        'plan_length_typical': 'O(n/k)',
        'key_challenge': 'Utilisation des pinces'
    },
    'Logistics': {
        'objects_type': 'Heterogene (colis, vehicules, lieux)',
        'actions_per_object': 4,
        'state_size_formula': 'O(n x m x l)',
        'plan_length_typical': 'Variable',
        'key_challenge': 'Coordination multi-agent'
    },
    'Ferry': {
        'objects_type': 'Mixte (voitures, lieu)',
        'actions_per_object': 3,
        'state_size_formula': 'O(n)',
        'plan_length_typical': 'O(n^2)',
        'key_challenge': 'Sequencement optimal'
    },
    'Hanoi': {
        'objects_type': 'Homogene (disques, pegs)',
        'actions_per_object': 2,
        'state_size_formula': 'O(3^n)',
        'plan_length_typical': 'O(2^n)',
        'key_challenge': 'Recursivite'
    }
}

print("Analyse comparative des domaines classiques")
print("=" * 70)
for domain, props in domains_analysis.items():
    print(f"\n{domain}:")
    for key, value in props.items():
        print(f"  - {key}: {value}")

Analyse comparative des domaines classiques

Block World:
  - objects_type: Homogene (blocs)
  - actions_per_object: 4
  - state_size_formula: O(n^2)
  - plan_length_typical: O(n^2)
  - key_challenge: Ordre des empilages

Gripper:
  - objects_type: Mixte (balles, pinces)
  - actions_per_object: 3
  - state_size_formula: O(n x k)
  - plan_length_typical: O(n/k)
  - key_challenge: Utilisation des pinces

Logistics:
  - objects_type: Heterogene (colis, vehicules, lieux)
  - actions_per_object: 4
  - state_size_formula: O(n x m x l)
  - plan_length_typical: Variable
  - key_challenge: Coordination multi-agent

Ferry:
  - objects_type: Mixte (voitures, lieu)
  - actions_per_object: 3
  - state_size_formula: O(n)
  - plan_length_typical: O(n^2)
  - key_challenge: Sequencement optimal

Hanoi:
  - objects_type: Homogene (disques, pegs)
  - actions_per_object: 2
  - state_size_formula: O(3^n)
  - plan_length_typical: O(2^n)
  - key_challenge: Recursivite


### 5.1 Efficacite des heuristiques par domaine

In [16]:
# Tableau d'efficacite des heuristiques (donnees empiriques)
heuristic_effectiveness = {
    'Domain': ['Block World', 'Gripper', 'Logistics', 'Ferry', 'Hanoi'],
    'h_add': ['Moyenne', 'Bonne', 'Bonne', 'Moyenne', 'Faible'],
    'h_max': ['Faible', 'Moyenne', 'Moyenne', 'Faible', 'Faible'],
    'h_FF': ['Bonne', 'Tres bonne', 'Tres bonne', 'Bonne', 'Moyenne'],
    'LM-cut': ['Tres bonne', 'Bonne', 'Bonne', 'Bonne', 'Bonne'],
    'Blind': ['Faible', 'Faible', 'Faible', 'Faible', 'Faible']
}

# Affichage
print("Efficacite des heuristiques par domaine")
print("-" * 60)
print(f"{'Domaine':<15} {'h_add':<10} {'h_max':<10} {'h_FF':<12} {'LM-cut':<10}")
print("-" * 60)
for i, domain in enumerate(heuristic_effectiveness['Domain']):
    print(f"{domain:<15} {heuristic_effectiveness['h_add'][i]:<10} "
          f"{heuristic_effectiveness['h_max'][i]:<10} "
          f"{heuristic_effectiveness['h_FF'][i]:<12} "
          f"{heuristic_effectiveness['LM-cut'][i]:<10}")
print("-" * 60)
print("\nRemarque : h_FF est generalement le meilleur compromis vitesse/qualite")

Efficacite des heuristiques par domaine
------------------------------------------------------------
Domaine         h_add      h_max      h_FF         LM-cut    
------------------------------------------------------------
Block World     Moyenne    Faible     Bonne        Tres bonne
Gripper         Bonne      Moyenne    Tres bonne   Bonne     
Logistics       Bonne      Moyenne    Tres bonne   Bonne     
Ferry           Moyenne    Faible     Bonne        Bonne     
Hanoi           Faible     Faible     Moyenne      Bonne     
------------------------------------------------------------

Remarque : h_FF est generalement le meilleur compromis vitesse/qualite


### 5.2 Explosion combinatoire

La taille de l'espace d'etats croit rapidement :

| Domaine | n=3 | n=5 | n=10 | n=20 |
|---------|-----|-----|------|------|
| Block World | 13 | 841 | 58.9M | astronomique |
| Gripper | 64 | 1.6K | 1.7M | 1.8B |
| Logistics | 81 | 3.9K | 1.0G | astronomique |
| Hanoi | 27 | 243 | 59K | 3.5B |

> **Conclusion** : Les heuristiques sont essentielles pour explorer efficacement ces espaces.

---

## 6. Conseils pratiques de modelisation

Cette section presente les bonnes pratiques pour ecrire des domaines PDDL efficaces.

### 6.1 Bonnes pratiques

In [17]:
# Bonnes pratiques de modelisation PDDL
best_practices = [
    {
        'titre': 'Utiliser le typage',
        'description': 'Reduit l espace de recherche en contraignant les parametres',
        'exemple': '(:types truck - vehicle) plutot que verifier dans les preconditions'
    },
    {
        'titre': 'Minimiser les predicats',
        'description': 'Chaque predicat augmente la taille de l etat',
        'exemple': 'Deriver (empty ?v) de (not (exists (?p) (in ?p ?v))) plutot que stocker'
    },
    {
        'titre': 'Actions atomiques',
        'description': 'Une action = une transformation elementaire',
        'exemple': 'load et unload separement plutot que move-with-cargo'
    },
    {
        'titre': 'Nommage explicite',
        'description': 'Facilite le debug et la lecture des plans',
        'exemple': 'drive-truck plutot que move2'
    },
    {
        'titre': 'Tester avec petits instances',
        'description': 'Valider le domaine sur 2-3 objets avant de scaler',
        'exemple': 'Commencer avec 2 blocs, 2 lieux avant 20 blocs'
    }
]

print("Bonnes pratiques de modelisation PDDL")
print("=" * 70)
for i, practice in enumerate(best_practices, 1):
    print(f"\n{i}. {practice['titre']}")
    print(f"   Description: {practice['description']}")
    print(f"   Exemple: {practice['exemple']}")

Bonnes pratiques de modelisation PDDL

1. Utiliser le typage
   Description: Reduit l espace de recherche en contraignant les parametres
   Exemple: (:types truck - vehicle) plutot que verifier dans les preconditions

2. Minimiser les predicats
   Description: Chaque predicat augmente la taille de l etat
   Exemple: Deriver (empty ?v) de (not (exists (?p) (in ?p ?v))) plutot que stocker

3. Actions atomiques
   Description: Une action = une transformation elementaire
   Exemple: load et unload separement plutot que move-with-cargo

4. Nommage explicite
   Description: Facilite le debug et la lecture des plans
   Exemple: drive-truck plutot que move2

5. Tester avec petits instances
   Description: Valider le domaine sur 2-3 objets avant de scaler
   Exemple: Commencer avec 2 blocs, 2 lieux avant 20 blocs


### 6.2 Erreurs courantes

In [18]:
# Erreurs courantes a eviter
common_mistakes = [
    {
        'erreur': 'Oublier de supprimer les preconditions dans les effets',
        'exemple': '(precondition (at ?x ?l)) -> oublier (not (at ?x ?l)) dans effects',
        'consequence': 'L objet existe a deux endroits simultanement'
    },
    {
        'erreur': 'Conditions cycliques',
        'exemple': 'A requires B, B requires A',
        'consequence': 'Aucun plan possible (deadlock)'
    },
    {
        'erreur': 'Types mal hierarchises',
        'exemple': 'truck et car sans parent commun',
        'consequence': 'Actions dupliquees pour chaque type'
    },
    {
        'erreur': 'Buts inatteignables',
        'exemple': '(goal (and (on a b) (on b a)))',
        'consequence': 'Planificateur tourne indefiniment'
    },
    {
        'erreur': 'Predicats derives non maintenus',
        'exemple': 'clear(x) oublie quand on pose quelque chose sur x',
        'consequence': 'Etats incoherents, plans invalides'
    }
]

print("Erreurs courantes en modelisation PDDL")
print("=" * 70)
for i, mistake in enumerate(common_mistakes, 1):
    print(f"\n{i}. {mistake['erreur']}")
    print(f"   Exemple: {mistake['exemple']}")
    print(f"   Consequence: {mistake['consequence']}")

Erreurs courantes en modelisation PDDL

1. Oublier de supprimer les preconditions dans les effets
   Exemple: (precondition (at ?x ?l)) -> oublier (not (at ?x ?l)) dans effects
   Consequence: L objet existe a deux endroits simultanement

2. Conditions cycliques
   Exemple: A requires B, B requires A
   Consequence: Aucun plan possible (deadlock)

3. Types mal hierarchises
   Exemple: truck et car sans parent commun
   Consequence: Actions dupliquees pour chaque type

4. Buts inatteignables
   Exemple: (goal (and (on a b) (on b a)))
   Consequence: Planificateur tourne indefiniment

5. Predicats derives non maintenus
   Exemple: clear(x) oublie quand on pose quelque chose sur x
   Consequence: Etats incoherents, plans invalides


### 6.3 Debugging PDDL

In [19]:
# Outils et techniques de debugging PDDL
debug_techniques = [
    "1. Valider la syntaxe avec un validateur PDDL (ex: validator de Valencia)",
    "2. Executer le plan a la main sur l etat initial",
    "3. Verifier que chaque but est accessible individuellement",
    "4. Reduire le probleme (moins d objets) pour isoler le bug",
    "5. Afficher l etat apres chaque action (mode debug)",
    "6. Utiliser un planificateur avec messages d erreur detailles",
    "7. Comparer avec un domaine similaire connu fonctionnel"
]

print("Techniques de debugging PDDL")
print("=" * 50)
for technique in debug_techniques:
    print(f"  {technique}")

Techniques de debugging PDDL
  1. Valider la syntaxe avec un validateur PDDL (ex: validator de Valencia)
  2. Executer le plan a la main sur l etat initial
  3. Verifier que chaque but est accessible individuellement
  4. Reduire le probleme (moins d objets) pour isoler le bug
  5. Afficher l etat apres chaque action (mode debug)
  6. Utiliser un planificateur avec messages d erreur detailles
  7. Comparer avec un domaine similaire connu fonctionnel


## Exercice : Modelisation d'un domaine de planification personnalise

### Contexte : Entrepot automatise

Un entrepot utilise un robot pour deplacer des colis entre des emplacements. Le robot peut porter un seul colis a la fois.

**Objets :**
- Emplacements : `depot`, `zone_a`, `zone_b`, `zone_c`
- Colis : `colis1`, `colis2`
- Robot : `robot1`

**Etat initial :**
- `robot1` est au `depot`, les bras vides
- `colis1` est en `zone_a`, `colis2` est en `zone_b`

**But :** Les deux colis doivent etre au `depot`

**Actions PDDL a implementer :**
1. `deplacer(robot, de, vers)` : deplace le robot d'un emplacement a un autre
2. `charger(robot, colis, lieu)` : le robot prend un colis (robot et colis au meme endroit, robot libre)
3. `decharger(robot, colis, lieu)` : le robot depose un colis (robot porte le colis)

### Objectifs

1. Implementer le domaine avec `unified_planning` (fluents, actions, preconditions, effets)
2. Resoudre le probleme avec Fast Downward
3. Afficher le plan trouve et calculer le nombre d'etapes
4. (Bonus) Ajouter une contrainte de carburant (numerique) et resoudre avec un solver qui supporte les fluents numeriques

### Indices

- Fluents booleens : `at(robot, lieu)`, `holding(robot, colis)`, `at_box(colis, lieu)`, `free(robot)`
- `up.BoolType()` pour les fluents booleens, `up.UserType("Robot")` pour les types
- Pattern de resolution : `with up.OneshotPlanner(name='fast-downward') as planner: result = planner.solve(problem)`

In [20]:
# --- Exercice : Domaine Entrepot Automatise ---
import unified_planning as up
from unified_planning.shortcuts import *

# 1. Definir les types
# Location = UserType("Location")
# Robot = UserType("Robot")
# Box = UserType("Box")
pass  # TODO: A vous de jouer !

# 2. Definir les fluents booleens
# at = Fluent("at", BoolType(), robot=Robot, location=Location)
# at_box = Fluent("at_box", BoolType(), box=Box, location=Location)
# holding = Fluent("holding", BoolType(), robot=Robot, box=Box)
# free = Fluent("free", BoolType(), robot=Robot)

# 3. Definir les objets
# depot = Object("depot", Location)
# zone_a = Object("zone_a", Location)
# ...

# 4. Definir les actions (deplacer, charger, decharger)
# Chaque action a : parameters, preconditions, effects
# move = InstantaneousAction("deplacer", r=Robot, src=Location, dst=Location)
# move.add_precondition(at(move.r, move.src))
# move.add_effect(at(move.r, move.src), False)
# move.add_effect(at(move.r, move.dst), True)

# 5. Construire le probleme
# problem = Problem("entrepot")
# problem.add_fluent(at, default_initial_value=False)
# ...
# problem.set_initial_value(at(robot1, depot), True)
# problem.set_initial_value(free(robot1), True)
# problem.add_goal(at_box(colis1, depot))
# problem.add_goal(at_box(colis2, depot))

# 6. Resoudre
# with OneshotPlanner(name="fast-downward") as planner:
#     result = planner.solve(problem)
#     if result.plan:
#         print(f"Plan trouve : {len(result.plan.action_instances)} etapes")
#         for step in result.plan.action_instances:
#             print(f"  {step}")
#     else:
#         print("Pas de plan trouve")

---

## 7. Resume et conclusions

### 7.1 Points cles du notebook

| Concept | Description |
|---------|-------------|
| **Block World** | Domaine de reference pour la manipulation, complexite O(n^2) |
| **Logistics** | Transport multi-agent, coordination camions/avions |
| **Gripper** | Robot avec pinces multiples, parallelisme possible |
| **Domaines classiques** | Benchmarks standardises pour comparer les planificateurs |
| **Modelisation PDDL** | Typage, actions atomiques, predicats minimaux |

### 7.2 Taxonomie des domaines

```
Domaines de planification
|-- Manipulation
|   |-- Block World (empiler/deseempiler)
|   |-- Hanoi (disques sur pegs)
|-- Transport
|   |-- Gripper (robot + pinces)
|   |-- Ferry (ferry + voitures)
|   |-- Logistics (multi-vehicules)
|-- Gestion
|   |-- Depots (entrepot + grues)
|   |-- Satellite (observations)
|-- Puzzle
    |-- Hanoi
    |-- 8-puzzle
```

### 7.3 Prochaines etapes

Dans le notebook **Planners-7-OR-Tools**, nous explorerons :
- La programmation par contraintes (CP-SAT)
- L'optimisation combinatoire
- La comparaison avec la planification classique

---

## Ressources

- [IPC Benchmarks](https://github.com/aibasel/downward-benchmarks) - Domaines PDDL standards
- [PDDL Reference](https://planning.wiki/) - Documentation complete
- [Fast Downward](https://www.fast-downward.org/) - Planificateur de reference
- [unified-planning](https://github.com/aiplan4eu/unified-planning) - Interface Python

---

**Notebook suivant** : [Planners-7-OR-Tools](../03-Advanced/Planners-7-OR-Tools.ipynb)